In [0]:
dbutils.widgets.text("Incremental_flag",'0')

In [0]:
Incremental_flag= dbutils.widgets.get("Incremental_flag")


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import * 

##Getting silver Data

In [0]:

df = spark.read.format('parquet').load('abfss://silver@retailstgaccpd.dfs.core.windows.net/cleaned_data/')
        

## fetching relative columns

In [0]:
df_src = spark.sql(''' 
SELECT DISTINCT (Model_ID) as Model_ID, Model_Category from parquet.`abfss://silver@retailstgaccpd.dfs.core.windows.net/cleaned_data`
''')

#Getting schema of source df with empty

### dim_model sink initial and incremental load

In [0]:
if spark.catalog.tableExists('cars_data_catalog.gold.dim_model'):
    df_sink= spark.sql('''
        SELECT dim_model_key, Model_ID, Model_category from cars_data_catalog.gold.dim_model
        ''')
else:
    df_sink= spark.sql('''
    SELECT 1 as dim_model_key, Model_ID, Model_category from parquet.`abfss://silver@retailstgaccpd.dfs.core.windows.net/cleaned_data` 
    where 1=0
    ''')


### Filtering new records vs new records

In [0]:
df_filtered = df_src.join(df_sink, df_src.Model_ID==df_sink.Model_ID,how='left').select(df_src['Model_ID'],df_src['Model_Category'], df_sink['dim_model_key'])

### df_filter_old

In [0]:
df_filter_old = df_filtered.filter(col('dim_model_key').isNotNull())

In [0]:
df_filter_new = df_filtered.filter(col('dim_model_key').isNull()).select(df_filtered['Model_ID'],df_filtered['Model_Category'])


### Create Surrogate Key

**get max value for surrogate key **

In [0]:
if (Incremental_flag== '0'):
    max_value=1
else:
    max_value_df = spark.sql("SELECT MAX(dim_model_key)  FROM cars_data_catalog.gold.dim_model")
    max_value = max_value_df.collect()[0][0]+1

**create a new surrogate key and add max value in it**

In [0]:
df_filter_new = df_filter_new.withColumn('dim_model_key', max_value + monotonically_increasing_id())

### Create Final df = filter_new + filter_old

In [0]:
df_final = df_filter_new.union(df_filter_old)

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists('cars_data_catalog.gold.dim_model'):
    delta_tbl= DeltaTable.forPath(spark, "abfss://gold@retailstgaccpd.dfs.core.windows.net/dim_model")

    delta_tbl.alias("trg").merge(df_final.alias("src"), "trg.dim_model_key = src.dim_model_key")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()

else:
    df_final.write.format("delta").\
    mode("overwrite")\
    .option("path", "abfss://gold@retailstgaccpd.dfs.core.windows.net/dim_model")\
    .saveAsTable('cars_data_catalog.gold.dim_model')

In [0]:
%sql
select * from cars_data_catalog.gold.dim_model;

Model_ID,Model_Category,dim_model_key
Hon-M219,Hon,1
Hyu-M161,Hyu,2
Toy-M101,Toy,3
Jee-M11,Jee,4
Tat-M192,Tat,5
Toy-M105,Toy,6
For-M16,For,7
Aud-M228,Aud,8
Nis-M81,Nis,9
Toy-M204,Toy,10
